# ② 검색 — 색인 만들기 · 검색
담당: 검색 · 코드: `rag/retrieval/indexing.py`, `rag/retrieval/retrieval.py`

01 전처리의 `artifacts/chunks.jsonl` (또는 v10 전처리 데이터셋의 청크)로 Dense(KURE-v1 임베딩 → FAISS)와 BM25(Kiwi 형태소) 색인을 만들어 `artifacts/index/` 에 저장합니다.
- 임베딩은 청크를 `embedding.max_seq_length`(2,048) 토큰까지 읽습니다 — 표 청크가 잘리지 않습니다.
- BM25 는 요구사항 코드(`SFR-001` · `SFR-상담-001` · `SFR-01-001`)를 한 토큰으로, 부정어 '않' · '아니' 도 남기고, 사용자 사전 · 별칭(부가세 → 부가가치세, 지체 상금 → 지체상금)으로 질문과 문서의 낱말을 맞춥니다.

검색은 두 점수를 섞은 하이브리드입니다.
- 질문의 요구사항 코드가 그대로 든 청크를 맨 앞에 둡니다.
- 질문에 발주기관이 나오면 그 기관 문서 안에서만 찾습니다. 기관명은 파일명 앞부분(`경기도 안양시_…`), 발주 기관 목록 CSV(`paths.org_csv`), 제목과 표지의 발주 · 주관 · 수요기관 칸에서 모으고, 띄어쓰기는 무시합니다('아시아 물 위원회' = '아시아물위원회').
  목록 CSV 가 없으면 파일명 앞부분으로 `data/data_list.csv` 를 자동으로 만듭니다. 이미 있으면 덮어쓰지 않습니다.
- 결과는 개수가 아니라 토큰 예산(`retrieval.token_budget`)으로 자르고, 같은 문서에서 숫자까지 같은 거의 같은 청크만 하나로 줄입니다.
  다른 사업의 같은 문구나 금액 · 기간만 다른 청크는 모두 근거로 남깁니다.

`.env` 에 Langfuse 키가 있으면 검색할 때마다 Langfuse Cloud 에 `retrieve-context` 로 기록됩니다 (없어도 똑같이 동작).

In [3]:
import pandas as pd

from rag.config import ARTIFACTS, ROOT, load_config, read_jsonl, write_jsonl
from rag.retrieval.indexing import build_index, load_v10
from rag.retrieval.retrieval import HybridIndex

cfg = load_config()

## 0. 입력 청크 고르기
`USE_V10 = True` 면 v10 전처리 데이터셋([mitguchin/-json-](https://github.com/mitguchin/-json-))의 청크를 씁니다.
01 과 같은 형식으로 `artifacts/chunks.jsonl` · `docs.jsonl` 에 저장하므로(01 결과를 덮어씀) 03 검증 ~ 05 평가도 같은 청크를 씁니다.
목차 청크, 그림 자리표시(`[IMAGE]`)만 있는 청크, `chunking.min_chunk_chars` 보다 짧은 청크('요구사항 분류 | 기능 요구사항' 같은 표 머리 줄)는 뺍니다.
`False` 면 01 전처리 결과를 그대로 씁니다. 바꾼 뒤에는 1번(색인 만들기)부터 다시 실행하세요.

In [4]:
USE_V10 = True
V10_DIR = ROOT / "data"  # mitguchin/-json- 을 받은 폴더 (final_elements.jsonl · final_chunks.jsonl)

if USE_V10:
    v10_chunks, v10_docs = load_v10(V10_DIR, cfg["chunking"]["min_chunk_chars"])
    write_jsonl(v10_chunks, ARTIFACTS / "chunks.jsonl")
    write_jsonl(v10_docs, ARTIFACTS / "docs.jsonl")
    print(f"v10 청크 {len(v10_chunks)}개 · 문서 {len(v10_docs)}개 → artifacts/chunks.jsonl · docs.jsonl")

v10 청크 15339개 · 문서 98개 → artifacts/chunks.jsonl · docs.jsonl


## 1. 색인 만들기
`임베딩에서 잘린 청크` 가 0 이면 모든 청크가 끝까지 임베딩된 것입니다.

In [5]:
chunks = read_jsonl(ARTIFACTS / "chunks.jsonl")
build_index(chunks, cfg, ARTIFACTS / "index")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/480 [00:00<?, ?it/s]

{'청크': 15339,
 '벡터 차원': 1024,
 'BM25 토큰': 2315544,
 'BM25 어휘': 18531,
 '임베딩에서 잘린 청크(>2048 토큰)': 0}

## 2. 검색해 보기
`config.yaml` 의 `retrieval` (token_budget · dense_weight · candidate_k)을 바꿔 가며 결과를 확인하세요.

In [6]:
index = HybridIndex(ARTIFACTS, cfg)
QUESTION = "고려대학교 차세대 포털 시스템 사업의 응답 시간 요구사항은?"
pd.DataFrame(index.search(QUESTION))[["score", "title", "page", "kind", "section", "body"]]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 


발주 기관 목록이 없어 문서 표지 · 제목으로 만들었습니다: /home/spai1322/rfg_rag/data/data_list.csv


,score,title,page,kind,section,body
0,0.8075,고려대학교 차세대 포털·학사 정보시스템 구축사업,49,text,,PER-001 성능요건\nPER-002 온라인성 업무응답시간
1,0.7700,고려대학교 차세대 포털·학사 정보시스템 구축사업,1,text,,제 안 요 청 서\n고려대학교\n 차세대 포털·학사 정보시스템 구축 사업\n2024...
2,0.7292,고려대학교 차세대 포털·학사 정보시스템 구축사업,49,text,❏ 요구사항 목록표,PER-003 웹페이지 디스플레이시간\n 성능요구사항\n PER-004 오류응답시간...
3,0.6399,고려대학교 차세대 포털·학사 정보시스템 구축사업,175,text,,-175-\n요구사항 분류 성능 요구사항\n요구사항 번호 PER-002\n요구사항 ...
4,0.4113,고려대학교 차세대 포털·학사 정보시스템 구축사업,270,text,,개인정보 수집·이용 동의서\n개인정보 수집/이용 동의 안내\n개인정보보호법에 따라 ...
5,0.3491,고려대학교 차세대 포털·학사 정보시스템 구축사업,33,text,,❏ 사용자 편의성 향상 및 업무 효율성 극대화를 위한 시스템 통합 및 연계 강화\n...
6,0.2938,고려대학교 차세대 포털·학사 정보시스템 구축사업,45,text,❏ 요구사항 목록표,SFR-공통-008 전산관리\n 기능요구사항 SFR-포털-001 포털공통\n (SF...
7,0.2173,고려대학교 차세대 포털·학사 정보시스템 구축사업,50,text,❏ 요구사항 목록표,PMR-004 사업관리 방법론 프로젝트관리\n 요구사항 PMR-005 프로젝트 수행...
8,0.2076,고려대학교 차세대 포털·학사 정보시스템 구축사업,200,table,,"산출정보 • 데이터검증계획서, 데이터검증결과서\n요구사항 분류 | 데이터 요구사항"
9,0.1841,고려대학교 차세대 포털·학사 정보시스템 구축사업,174,table,,"산출정보 • 프로그램 목록, 프로그램명세서, 화면정의서, 사용자매뉴얼 등\n❏ 성능..."


## 3. 기관 라우팅 점검
문서마다 연결된 기관명입니다. `기관명` 이 빈 문서는 질문에 기관명이 나와도 전체 문서에서 찾습니다.

In [7]:
doc_orgs = {d: sorted(o for o, ids in index.router.org_docs.items() if d in ids) for d in index.router.titles}
print(f"라우팅되는 문서 {sum(map(bool, doc_orgs.values()))}/{len(doc_orgs)}")
pd.DataFrame({"문서": [index.router.titles[d] for d in doc_orgs], "기관명": [", ".join(o) for o in doc_orgs.values()]})

라우팅되는 문서 98/98


,문서,기관명
0,사단법인아시아물위원회사무국 우즈벡-키르기즈스탄 기후변화대응 스,"아시아물위원회, 아시아물위원회사무국"
1,서민금융진흥원 서민금융진흥원 서민금융 채팅 상담시스템 구축,서민금융진흥원
2,한국철도공사 (용역) [재공고][긴급][협상형]운행정보기록 자동분석시스,한국철도공사
3,한국농어촌공사 아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아,한국농어촌공사
4,한국연구재단 2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선,한국연구재단
...,...,...
93,대전대학교 대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전,대전대학교
94,축산물품질평가원 꿀 품질평가 전산시스템 기능개선 사업,축산물품질평가원
95,한국재정정보원 e나라도움 업무시스템 웹 접근성 컨설팅,한국재정정보원
96,국가과학기술지식정보서비스 통합정보시스템 고도화 용역,국가과학기술지식정보서비스


질문 하나가 어느 기관 문서로 좁혀지는지 봅니다. `{}` 이면 전체 문서에서 찾습니다.

In [8]:
ROUTE_QUESTION = "경기도 안양시 배드민턴장 예약시스템 사업 기간은?"
{org: [index.router.titles[d] for d in ids] for org, ids in index.router.route(ROUTE_QUESTION).items()}

{'경기도 안양시': ['경기도 안양시 호계체육관 배드민턴장 및 탁구장 예약시스템 구축 용역']}

## 4. 발주 기관 목록
`data/data_list.csv` 내용입니다. `발주 기관` 이 비었거나 틀린 칸은 직접 고쳐 저장한 뒤 2번(검색해 보기)부터 다시 실행하세요.
예전 방식(표지만 보고)으로 만들어져 빈칸이 많은 파일이면, 지우고 2번부터 다시 실행하면 파일명 기준으로 새로 만듭니다.

In [9]:
pd.read_csv(ROOT / cfg["paths"]["org_csv"], encoding="utf-8-sig")

,파일명,발주 기관
0,사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp,아시아물위원회사무국
1,서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp,서민금융진흥원
2,한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp,한국철도공사
3,한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp,한국농어촌공사
4,한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp,한국연구재단
...,...,...
93,대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp,대전대학교
94,축산물품질평가원_꿀 품질평가 전산시스템 기능개선 사업.hwp,축산물품질평가원
95,한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp,한국재정정보원
96,국가과학기술지식정보서비스_통합정보시스템 고도화 용역.hwp,국가과학기술지식정보서비스
